In [1]:
# === SETUP: Run this first! ===
import os
import sys

# Change to project root and add to Python path
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)  # Goes up one level from 'notebooks/'
os.chdir(project_root)
sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"tsnn module path: {os.path.join(project_root, 'tsnn')}")

Project root: /Users/gremy/Code/TSNN-1
tsnn module path: /Users/gremy/Code/TSNN-1/tsnn


In [2]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV, RidgeCV, LinearRegression
import torch
from torch.utils.data import Dataset, random_split
from torch.utils.data import DataLoader
import importlib
import sys
sys.path.append('/Users/cyrilgarcia/notebooks/tsnn/')

import tsnn

from tsnn.generators import generators
from tsnn import tstorch
from tsnn.benchmarks import benchmark_comparison, ml_benchmarks, torch_benchmarks
from tsnn import utils
from tsnn.tstorch import transformers
import torch.nn.functional as F
import math
from typing import Optional

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

from torch import nn
from tqdm import tqdm
device = 'mps'


plt.style.use('ggplot')

In [3]:
# This is the new version of the notebook to make all the figures for the paper.

In [4]:
from tsnn.tstorch import models
from tsnn.benchmarks import torch_benchmarks
from tsnn.benchmarks.torch_benchmarks import MSELossWithL1Sparsity

In [5]:
# Notebook to run all the experiments for the paper.

# Setup

For now we will be working with the following effects:
- Simple linear dependency
- TS shift with lags chosen randomly for each feature, but constant accros stocks
- CS shift, with lags chosen randomly for each stock and each feature
- TS-CS shift: combination of the two lags above
- Conditioning of one feature by another

We will fix the following global correlation levels: $\rho$ = 1,2,5,10,20,50. The first and last might not be too meaningful.

Next let's fix the size of the data that we want to test. We will always work with $T=4000$ points, and use $2500$ for training, $1500$ for testing. For the other dimensions:
- Default 1: n_ts = 10, n_rolling = 10, n_f = 20. This gives $\gamma = 0.8$.
- Default 2: n_ts = 10, n_rolling = 10, n_f = 5. This gives $\gamma = 0.2$.

Could also add two extra cases to test larger n_rolling or n_ts:
- Long TS: n_ts = 10, n_rolling = 40, n_f = 5. This gives $\gamma = 0.8$.
- Long CS: n_ts = 40, n_rolling = 10, n_f = 5. This gives $\gamma = 0.8$.

Can also run an experiment varying jointly $\rho$ and $\gamma$ at a fixed theo correl level.

Next we need to compute a $\rho$ per feature. In the case where n_f=5, we will simply take $\frac{\rho}{\sqrt{5}}$ per feature. In the case n_f=20, we will take $\frac{\rho}{\sqrt{10}}$ for half of the features and $0$ for the half.

In [6]:
# Some basic functions.

In [7]:
def causal_mask(b, h, q_idx, kv_idx):
    return q_idx >= kv_idx

def causal_mask_radius(b, h, q_idx, kv_idx):
    return (q_idx >= kv_idx) & (q_idx <= kv_idx+1)

def plot_mask(mask_fn, seq_len=20, title=None, device="cpu"):
    """
    Plot a binary attention mask defined by mask_fn(b,h,q_idx,kv_idx)
    as a (seq_len x seq_len) matrix.
    """
    q_idx = torch.arange(seq_len, device=device)
    kv_idx = torch.arange(seq_len, device=device)
    b = torch.zeros(1, device=device) 
    h = torch.zeros(1, device=device)

    mask = mask_fn(b, h, q_idx[:, None], kv_idx[None, :])
    mask = mask.float().cpu()

    return pd.DataFrame(mask).style.background_gradient(axis=None).format(precision=0)

def build_attention_mask(mask_fn, seq_len, device="cpu"):
    q_idx = torch.arange(seq_len, device=device)
    kv_idx = torch.arange(seq_len, device=device)
    b = torch.zeros(1, device=device)
    h = torch.zeros(1, device=device)
    mask_bool = mask_fn(b, h, q_idx[:, None], kv_idx[None, :])  # (seq_len, seq_len)
    return mask_bool

In [8]:
def get_ols_corr(N_fea, T_train, rho, oos=True):
    ratio = N_fea/T_train
    if oos:
        return rho / np.sqrt(rho**2 + (1-rho**2) * ratio/(1-ratio))
    else:
        return rho / np.sqrt(rho**2 + (1-rho**2) * ratio)


In [9]:
def keep_topk_per_row3(x, k=3):
    vals, idx = torch.topk(x, k=k, dim=-1, largest=True)
    out = torch.zeros_like(x)
    out.scatter_(-1, idx, 1)
    return out

In [10]:
def keep_by_max_value2(x, frac=0.2):
    row_max = x.max(dim=-1, keepdim=True).values
    threshold = frac * row_max
    out = (x >= threshold).to(x.dtype)
    return out


# Run of all models on all effects

In [11]:
def run_models1(rhos, effect, T=4000, n_ts=10, n_f=5, n_rolling=10):

    list_effects1 = [effect]*n_f

    if n_f >=10:
        half_n_f = int(n_f/2)
        correl_split_by_fea1 = [1/np.sqrt(half_n_f)]*(half_n_f) + [0]*half_n_f 
    else:
        correl_split_by_fea1 = [1/np.sqrt(n_f)]*n_f

    mask = causal_mask
    mask_c = build_attention_mask(mask, n_rolling, device=device)
    
    z = generators.Generator(T, n_ts, n_f)
    
    res_train = []
    res_test = []

    for (i,rho) in enumerate(rhos):
        print("Running correl level:", rho)
        theo_correl_is = get_ols_corr(n_ts*n_f*n_rolling, T*0.625, rho, oos=False)
        theo_correl_oos = get_ols_corr(n_ts*n_f*n_rolling, T*0.625, rho, oos=True)


        z.generate_dataset_gr_simple(
            global_corr=rho, 
            correl_split_by_fea=correl_split_by_fea1, 
            list_type_effects=list_effects1,
            list_type_interaction=["cond"], 
            random_ts_shift=n_rolling,
        )


        models_torch = {
        'Global_MLP': models.GlobalMLP(n_ts, n_f, n_rolling, dropout=0.1).to(device),
        '1D_Trans_TT': models.OneDimensionalTransformer(n_ts, n_f, n_rolling, mask=mask_c, attn_direction="T",  num_attn_layers=2,
                                        d_model=64, dim_feedforward=256, nhead=8, compression="MLP", num_mlp_layers=2,
                                        dropout=0.1, roll_y=True).to(device),
        '1D_Trans_CC': models.OneDimensionalTransformer(n_ts, n_f, n_rolling, mask=mask_c, attn_direction="C",  num_attn_layers=2,
                                        d_model=64, dim_feedforward=256, nhead=8, compression="MLP", num_mlp_layers=2,
                                        dropout=0.1,  roll_y=False).to(device),                                
        '2D_Trans_TC': models.CustomBiDimensionalTransformer(
            n_ts, n_f, n_rolling, mask=mask_c, layers='TC', nhead=8,
            dropout=0.1, d_model=64, dim_feedforward=256, sparsify=None, roll_y=True, embeddings="both",
        ).to(device),
        '2D_Trans_TCTC': models.CustomBiDimensionalTransformer(
            n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
            dropout=0.1, d_model=64, dim_feedforward=256, sparsify=None, roll_y=True, embeddings="both",
        ).to(device),
        '1D_Trans_TT_sparse': models.OneDimensionalTransformer(n_ts, n_f, n_rolling, mask=mask_c, attn_direction="T",  num_attn_layers=2,
                                        d_model=64, dim_feedforward=256, nhead=8, compression="MLP", num_mlp_layers=2,
                                        dropout=0.1, roll_y=True, sparsify=keep_topk_per_row3,).to(device),
        '1D_Trans_CC_sparse': models.OneDimensionalTransformer(n_ts, n_f, n_rolling, mask=mask_c, attn_direction="C",  num_attn_layers=2,
                                        d_model=64, dim_feedforward=256, nhead=8, compression="MLP", num_mlp_layers=2,
                                        dropout=0.1,  roll_y=False, sparsify=keep_topk_per_row3,).to(device),                                
        '2D_Trans_TC_sparse': models.CustomBiDimensionalTransformer(
            n_ts, n_f, n_rolling, mask=mask_c, layers='TC', nhead=8,
            dropout=0.1, d_model=64, dim_feedforward=256, sparsify=keep_topk_per_row3, roll_y=True, embeddings="both",
        ).to(device),
        '2D_Trans_TCTC_sparse': models.CustomBiDimensionalTransformer(
            n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
            dropout=0.1, d_model=64, dim_feedforward=256, sparsify=keep_topk_per_row3, roll_y=True, embeddings="both",
        ).to(device),
        }

        models_torch = {
            k: torch_benchmarks.TorchWrapper(
                models_torch[k], 
                optimizer=torch.optim.AdamW(models_torch[k].parameters(), lr=0.001),
                loss_fn=nn.MSELoss()  
            ) for k in models_torch
        }

        
        for k in models_torch:
            roll_y = True
            if k in ['Global_MLP', '1D_Trans_CC', '1D_Trans_CC_sparse']:
                roll_y = False
            z.get_dataloader(n_rolling=n_rolling, roll_y=roll_y)
            epochs=40
            if k in ['1D_Trans_CC', '1D_Trans_CC_sparse']:
                epochs=80
            models_torch[k].fit(z.train, test=z.test, epochs=epochs, plot=False, verbose=0)
            
      
        comp = benchmark_comparison.Comparator(models=[models_torch[k] for k in models_torch], 
        model_names=[k for k in models_torch]
        )

        corr_train = comp.correl(z, mode="train", return_values=True)
        corr_test  = comp.correl(z, mode="test",  return_values=True)

        for k in models_torch:

            train_corr = corr_train.loc[k, "optimal"]
            test_corr  = corr_test.loc[k, "optimal"]

            res_train.append({
                "rho": rhos[i],
                "model": k,
                "train_corr_optimal": train_corr
            })
            res_test.append({
                "rho": rhos[i],
                "model": k,
                "test_corr_optimal": test_corr
            })


        res_train.append({
                "rho": rhos[i],
                "model": "theo_correl",
                "train_corr_optimal": theo_correl_is
            })
        res_test.append({
                "rho": rhos[i],
                "model": "theo_correl",
                "test_corr_optimal": theo_correl_oos
            })

    
    res_train = pd.DataFrame(res_train)
    res_test  = pd.DataFrame(res_test)

    res_train = res_train.pivot(index="rho", columns="model", values="train_corr_optimal")
    res_test  = res_test.pivot(index="rho", columns="model", values="test_corr_optimal")

    return res_train, res_test

## n_f = 5

In [ ]:
#all_effects_nf5_train_dic = {}
#all_effects_nf5_test_dic = {}

In [13]:
for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift', 'TS_cond', 'CS_cond']:
    print("Running_effect:", effect)
    out_train1, out_test1 = run_models1([0.02, 0.05, 0.1, 0.2, 0.5], effect, T=4000, n_ts=10, n_f=5, n_rolling=10)
    all_effects_nf5_train_dic[effect] = out_train1
    all_effects_nf5_test_dic[effect] = out_test1

Running_effect: lin
Running correl level: 0.02


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.05


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.1


...

/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.2


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.5


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [14]:
for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift', 'TS_cond', 'CS_cond']:
    print(effect)
    display(all_effects_nf5_train_dic[effect].T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
    display(all_effects_nf5_test_dic[effect].T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

lin


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
1D_Trans_CC,0.037,0.091,0.142,0.278,0.640
1D_Trans_CC_sparse,0.039,0.070,0.174,0.283,0.687
1D_Trans_TT,0.043,0.091,0.180,0.324,0.713
1D_Trans_TT_sparse,0.043,0.094,0.184,0.337,0.711
2D_Trans_TC,0.091,0.171,0.413,0.576,0.916
2D_Trans_TCTC,0.098,0.216,0.355,0.452,0.883
2D_Trans_TCTC_sparse,0.091,0.230,0.263,0.517,0.907
2D_Trans_TC_sparse,0.084,0.228,0.344,0.660,0.958
Global_MLP,0.030,0.052,0.110,0.201,0.504


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
1D_Trans_CC,0.048,0.117,0.176,0.329,0.699
1D_Trans_CC_sparse,0.043,0.068,0.208,0.344,0.732
1D_Trans_TT,0.041,0.097,0.196,0.353,0.728
1D_Trans_TT_sparse,0.050,0.104,0.193,0.364,0.728
2D_Trans_TC,0.090,0.161,0.411,0.578,0.917
2D_Trans_TCTC,0.101,0.214,0.370,0.439,0.886
2D_Trans_TCTC_sparse,0.072,0.227,0.269,0.527,0.908
2D_Trans_TC_sparse,0.085,0.224,0.334,0.664,0.959
Global_MLP,0.049,0.086,0.163,0.312,0.651


TS_shift


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
1D_Trans_CC,0.040,0.078,0.151,0.295,0.661
1D_Trans_CC_sparse,0.041,0.075,0.154,0.298,0.679
1D_Trans_TT,0.015,0.042,0.078,0.178,0.444
1D_Trans_TT_sparse,0.012,0.042,0.081,0.162,0.433
2D_Trans_TC,0.024,0.118,0.054,0.442,0.833
2D_Trans_TCTC,0.022,0.091,0.075,0.392,0.828
2D_Trans_TCTC_sparse,0.022,0.128,0.171,0.450,0.841
2D_Trans_TC_sparse,0.033,0.174,0.245,0.545,0.872
Global_MLP,0.024,0.054,0.103,0.208,0.509


...

rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
1D_Trans_CC,0.028,0.037,0.095,0.185,0.559
1D_Trans_CC_sparse,0.016,0.039,0.095,0.189,0.572
1D_Trans_TT,0.011,0.026,0.076,0.147,0.386
1D_Trans_TT_sparse,0.014,0.032,0.074,0.141,0.378
2D_Trans_TC,0.010,0.013,0.031,0.080,0.411
2D_Trans_TCTC,0.010,0.015,-0.003,0.066,0.390
2D_Trans_TCTC_sparse,-0.010,0.023,0.037,0.083,0.398
2D_Trans_TC_sparse,-0.001,0.011,0.019,0.073,0.339
Global_MLP,0.021,0.041,0.102,0.190,0.504


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
1D_Trans_CC,0.018,0.016,0.028,0.073,0.421
1D_Trans_CC_sparse,0.028,0.022,0.035,0.082,0.437
1D_Trans_TT,0.005,0.009,0.006,0.002,0.006
1D_Trans_TT_sparse,-0.002,-0.007,0.006,-0.003,0.003
2D_Trans_TC,0.012,0.003,0.031,-0.001,0.307
2D_Trans_TCTC,-0.012,-0.000,0.011,0.004,0.273
2D_Trans_TCTC_sparse,0.005,0.015,0.043,0.031,0.248
2D_Trans_TC_sparse,-0.006,-0.004,0.023,0.040,0.267
Global_MLP,-0.001,0.008,0.012,0.010,0.018


CS_cond


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
1D_Trans_CC,0.016,0.046,0.084,0.178,0.425
1D_Trans_CC_sparse,0.012,0.047,0.088,0.175,0.407
1D_Trans_TT,0.017,0.052,0.086,0.179,0.447
1D_Trans_TT_sparse,0.017,0.050,0.082,0.185,0.446
2D_Trans_TC,0.010,0.017,0.066,0.139,0.438
2D_Trans_TCTC,0.022,0.034,0.078,0.064,0.407
2D_Trans_TCTC_sparse,0.005,0.042,0.063,0.153,0.496
2D_Trans_TC_sparse,0.010,0.034,0.041,0.095,0.447
Global_MLP,0.020,0.051,0.100,0.206,0.500


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
1D_Trans_CC,-0.008,-0.009,-0.008,0.004,0.005
1D_Trans_CC_sparse,0.011,-0.009,0.013,-0.005,0.001
1D_Trans_TT,-0.005,-0.005,0.007,0.051,0.113
1D_Trans_TT_sparse,0.007,0.028,0.015,0.035,0.118
2D_Trans_TC,-0.011,0.002,0.016,0.072,0.349
2D_Trans_TCTC,0.026,0.025,0.041,0.028,0.248
2D_Trans_TCTC_sparse,0.002,0.030,0.026,0.078,0.380
2D_Trans_TC_sparse,-0.001,0.012,0.022,0.052,0.339
Global_MLP,-0.011,-0.010,-0.001,-0.004,0.010


### Hardcoded df

In [15]:
# Here let's give the python code to define a pandas table that has exactly the above parameters.

In [48]:
# Generate code to recreate the DataFrame with rounded values
def generate_dataframe_code(df, decimals=3, var_name='df'):
    """Generate Python code to recreate a DataFrame with rounded floats."""
    
    # Round the DataFrame
    df_rounded = df.round(decimals)
    
    # Convert to dictionary format
    data_dict = df_rounded.to_dict('list')
    
    # Format the code string
    code = f"{var_name} = pd.DataFrame({{\n"
    for col, values in data_dict.items():
        # Replace NaN values with np.nan for proper representation
        formatted_values = []
        for val in values:
            if pd.isna(val):
                formatted_values.append('np.nan')
            else:
                formatted_values.append(repr(val))
        
        values_str = '[' + ', '.join(formatted_values) + ']'
        code += f"    '{col}': {values_str},\n"
    code += "})"
    
    return code


In [22]:
for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift', 'TS_cond', 'CS_cond']:
    print(generate_dataframe_code(all_effects_nf5_test_dic[effect], var_name="df_test_" + effect))


df_test_lin = pd.DataFrame({
    '1D_Trans_CC': [0.048, 0.117, 0.176, 0.329, 0.699],
    '1D_Trans_CC_sparse': [0.043, 0.068, 0.208, 0.344, 0.732],
    '1D_Trans_TT': [0.041, 0.097, 0.196, 0.353, 0.728],
    '1D_Trans_TT_sparse': [0.05, 0.104, 0.193, 0.364, 0.728],
    '2D_Trans_TC': [0.09, 0.161, 0.411, 0.578, 0.917],
    '2D_Trans_TCTC': [0.101, 0.214, 0.37, 0.439, 0.886],
    '2D_Trans_TCTC_sparse': [0.072, 0.227, 0.269, 0.527, 0.908],
    '2D_Trans_TC_sparse': [0.085, 0.224, 0.334, 0.664, 0.959],
    'Global_MLP': [0.049, 0.086, 0.163, 0.312, 0.651],
    'theo_correl': [0.04, 0.1, 0.197, 0.378, 0.756],
})
df_test_TS_shift = pd.DataFrame({
    '1D_Trans_CC': [0.041, 0.091, 0.178, 0.342, 0.71],
    '1D_Trans_CC_sparse': [0.05, 0.067, 0.183, 0.352, 0.721],
    '1D_Trans_TT': [0.005, 0.018, -0.004, 0.055, 0.16],
    '1D_Trans_TT_sparse': [0.009, 0.004, -0.008, 0.046, 0.141],
    '2D_Trans_TC': [0.007, 0.092, 0.028, 0.437, 0.819],
    '2D_Trans_TCTC': [0.012, 0.096, 0.05, 0.368, 0.81],


In [21]:
# Train dataframes

df_train_lin = pd.DataFrame({
    '1D_Trans_CC': [0.037, 0.091, 0.142, 0.278, 0.64],
    '1D_Trans_CC_sparse': [0.039, 0.07, 0.174, 0.283, 0.687],
    '1D_Trans_TT': [0.043, 0.091, 0.18, 0.324, 0.713],
    '1D_Trans_TT_sparse': [0.043, 0.094, 0.184, 0.337, 0.711],
    '2D_Trans_TC': [0.091, 0.171, 0.413, 0.576, 0.916],
    '2D_Trans_TCTC': [0.098, 0.216, 0.355, 0.452, 0.883],
    '2D_Trans_TCTC_sparse': [0.091, 0.23, 0.263, 0.517, 0.907],
    '2D_Trans_TC_sparse': [0.084, 0.228, 0.344, 0.66, 0.958],
    'Global_MLP': [0.03, 0.052, 0.11, 0.201, 0.504],
    'theo_correl': [0.045, 0.111, 0.219, 0.415, 0.791],
})
df_train_TS_shift = pd.DataFrame({
    '1D_Trans_CC': [0.04, 0.078, 0.151, 0.295, 0.661],
    '1D_Trans_CC_sparse': [0.041, 0.075, 0.154, 0.298, 0.679],
    '1D_Trans_TT': [0.015, 0.042, 0.078, 0.178, 0.444],
    '1D_Trans_TT_sparse': [0.012, 0.042, 0.081, 0.162, 0.433],
    '2D_Trans_TC': [0.024, 0.118, 0.054, 0.442, 0.833],
    '2D_Trans_TCTC': [0.022, 0.091, 0.075, 0.392, 0.828],
    '2D_Trans_TCTC_sparse': [0.022, 0.128, 0.171, 0.45, 0.841],
    '2D_Trans_TC_sparse': [0.033, 0.174, 0.245, 0.545, 0.872],
    'Global_MLP': [0.024, 0.054, 0.103, 0.208, 0.509],
    'theo_correl': [0.045, 0.111, 0.219, 0.415, 0.791],
})
df_train_CS_shift = pd.DataFrame({
    '1D_Trans_CC': [0.029, 0.045, 0.102, 0.19, 0.605],
    '1D_Trans_CC_sparse': [0.038, 0.049, 0.108, 0.213, 0.63],
    '1D_Trans_TT': [0.053, 0.081, 0.169, 0.318, 0.718],
    '1D_Trans_TT_sparse': [0.049, 0.079, 0.173, 0.318, 0.728],
    '2D_Trans_TC': [0.063, 0.093, 0.292, 0.526, 0.867],
    '2D_Trans_TCTC': [0.008, 0.124, 0.29, 0.451, 0.804],
    '2D_Trans_TCTC_sparse': [0.094, 0.107, 0.234, 0.472, 0.817],
    '2D_Trans_TC_sparse': [0.06, 0.092, 0.238, 0.474, 0.855],
    'Global_MLP': [0.032, 0.046, 0.1, 0.194, 0.519],
    'theo_correl': [0.045, 0.111, 0.219, 0.415, 0.791],
})
df_train_fea_cond = pd.DataFrame({
    '1D_Trans_CC': [0.02, 0.045, 0.099, 0.259, 0.6],
    '1D_Trans_CC_sparse': [0.012, 0.05, 0.095, 0.258, 0.607],
    '1D_Trans_TT': [0.018, 0.042, 0.08, 0.201, 0.465],
    '1D_Trans_TT_sparse': [0.022, 0.039, 0.074, 0.203, 0.449],
    '2D_Trans_TC': [0.082, 0.17, 0.247, 0.558, 0.833],
    '2D_Trans_TCTC': [0.095, 0.117, 0.19, 0.411, 0.779],
    '2D_Trans_TCTC_sparse': [0.032, 0.149, 0.214, 0.489, 0.808],
    '2D_Trans_TC_sparse': [0.069, 0.198, 0.276, 0.571, 0.854],
    'Global_MLP': [0.017, 0.039, 0.094, 0.22, 0.504],
    'theo_correl': [0.045, 0.111, 0.219, 0.415, 0.791],
})
df_train_TSCS_shift = pd.DataFrame({
    '1D_Trans_CC': [0.02, 0.042, 0.1, 0.213, 0.584],
    '1D_Trans_CC_sparse': [0.013, 0.035, 0.103, 0.214, 0.626],
    '1D_Trans_TT': [0.019, 0.029, 0.072, 0.184, 0.424],
    '1D_Trans_TT_sparse': [0.015, 0.028, 0.072, 0.169, 0.404],
    '2D_Trans_TC': [0.005, 0.006, 0.026, 0.06, 0.172],
    '2D_Trans_TCTC': [-0.008, 0.013, 0.024, 0.032, 0.217],
    '2D_Trans_TCTC_sparse': [-0.001, 0.011, 0.032, 0.036, 0.176],
    '2D_Trans_TC_sparse': [-0.002, 0.018, 0.025, 0.041, 0.118],
    'Global_MLP': [0.021, 0.04, 0.1, 0.202, 0.508],
    'theo_correl': [0.045, 0.111, 0.219, 0.415, 0.791],
})
df_train_TS_cond = pd.DataFrame({
    '1D_Trans_CC': [0.028, 0.037, 0.095, 0.185, 0.559],
    '1D_Trans_CC_sparse': [0.016, 0.039, 0.095, 0.189, 0.572],
    '1D_Trans_TT': [0.011, 0.026, 0.076, 0.147, 0.386],
    '1D_Trans_TT_sparse': [0.014, 0.032, 0.074, 0.141, 0.378],
    '2D_Trans_TC': [0.01, 0.013, 0.031, 0.08, 0.411],
    '2D_Trans_TCTC': [0.01, 0.015, -0.003, 0.066, 0.39],
    '2D_Trans_TCTC_sparse': [-0.01, 0.023, 0.037, 0.083, 0.398],
    '2D_Trans_TC_sparse': [-0.001, 0.011, 0.019, 0.073, 0.339],
    'Global_MLP': [0.021, 0.041, 0.102, 0.19, 0.504],
    'theo_correl': [0.045, 0.111, 0.219, 0.415, 0.791],
})
df_train_CS_cond = pd.DataFrame({
    '1D_Trans_CC': [0.016, 0.046, 0.084, 0.178, 0.425],
    '1D_Trans_CC_sparse': [0.012, 0.047, 0.088, 0.175, 0.407],
    '1D_Trans_TT': [0.017, 0.052, 0.086, 0.179, 0.447],
    '1D_Trans_TT_sparse': [0.017, 0.05, 0.082, 0.185, 0.446],
    '2D_Trans_TC': [0.01, 0.017, 0.066, 0.139, 0.438],
    '2D_Trans_TCTC': [0.022, 0.034, 0.078, 0.064, 0.407],
    '2D_Trans_TCTC_sparse': [0.005, 0.042, 0.063, 0.153, 0.496],
    '2D_Trans_TC_sparse': [0.01, 0.034, 0.041, 0.095, 0.447],
    'Global_MLP': [0.02, 0.051, 0.1, 0.206, 0.5],
    'theo_correl': [0.045, 0.111, 0.219, 0.415, 0.791],
})

In [23]:
# Test data

df_test_lin = pd.DataFrame({
    '1D_Trans_CC': [0.048, 0.117, 0.176, 0.329, 0.699],
    '1D_Trans_CC_sparse': [0.043, 0.068, 0.208, 0.344, 0.732],
    '1D_Trans_TT': [0.041, 0.097, 0.196, 0.353, 0.728],
    '1D_Trans_TT_sparse': [0.05, 0.104, 0.193, 0.364, 0.728],
    '2D_Trans_TC': [0.09, 0.161, 0.411, 0.578, 0.917],
    '2D_Trans_TCTC': [0.101, 0.214, 0.37, 0.439, 0.886],
    '2D_Trans_TCTC_sparse': [0.072, 0.227, 0.269, 0.527, 0.908],
    '2D_Trans_TC_sparse': [0.085, 0.224, 0.334, 0.664, 0.959],
    'Global_MLP': [0.049, 0.086, 0.163, 0.312, 0.651],
    'theo_correl': [0.04, 0.1, 0.197, 0.378, 0.756],
})
df_test_TS_shift = pd.DataFrame({
    '1D_Trans_CC': [0.041, 0.091, 0.178, 0.342, 0.71],
    '1D_Trans_CC_sparse': [0.05, 0.067, 0.183, 0.352, 0.721],
    '1D_Trans_TT': [0.005, 0.018, -0.004, 0.055, 0.16],
    '1D_Trans_TT_sparse': [0.009, 0.004, -0.008, 0.046, 0.141],
    '2D_Trans_TC': [0.007, 0.092, 0.028, 0.437, 0.819],
    '2D_Trans_TCTC': [0.012, 0.096, 0.05, 0.368, 0.81],
    '2D_Trans_TCTC_sparse': [0.024, 0.103, 0.173, 0.453, 0.831],
    '2D_Trans_TC_sparse': [0.018, 0.173, 0.228, 0.545, 0.864],
    'Global_MLP': [0.044, 0.075, 0.157, 0.325, 0.67],
    'theo_correl': [0.04, 0.1, 0.197, 0.378, 0.756],
})
df_test_CS_shift = pd.DataFrame({
    '1D_Trans_CC': [0.016, 0.007, 0.04, 0.094, 0.576],
    '1D_Trans_CC_sparse': [0.021, 0.02, 0.043, 0.128, 0.604],
    '1D_Trans_TT': [0.059, 0.092, 0.193, 0.349, 0.738],
    '1D_Trans_TT_sparse': [0.065, 0.095, 0.168, 0.324, 0.738],
    '2D_Trans_TC': [0.055, 0.08, 0.263, 0.519, 0.86],
    '2D_Trans_TCTC': [0.002, 0.114, 0.277, 0.423, 0.798],
    '2D_Trans_TCTC_sparse': [0.096, 0.094, 0.225, 0.46, 0.816],
    '2D_Trans_TC_sparse': [0.056, 0.079, 0.216, 0.459, 0.847],
    'Global_MLP': [0.037, 0.068, 0.16, 0.298, 0.678],
    'theo_correl': [0.04, 0.1, 0.197, 0.378, 0.756],
})
df_test_fea_cond = pd.DataFrame({
    '1D_Trans_CC': [0.018, 0.022, 0.047, 0.189, 0.524],
    '1D_Trans_CC_sparse': [-0.002, 0.024, 0.048, 0.184, 0.52],
    '1D_Trans_TT': [-0.013, 0.028, 0.019, 0.047, 0.165],
    '1D_Trans_TT_sparse': [0.03, 0.009, 0.019, 0.058, 0.152],
    '2D_Trans_TC': [0.051, 0.159, 0.236, 0.543, 0.811],
    '2D_Trans_TCTC': [0.08, 0.111, 0.169, 0.415, 0.757],
    '2D_Trans_TCTC_sparse': [0.038, 0.145, 0.214, 0.484, 0.787],
    '2D_Trans_TC_sparse': [0.061, 0.207, 0.275, 0.572, 0.842],
    'Global_MLP': [0.001, 0.012, -0.0, -0.002, 0.019],
    'theo_correl': [0.04, 0.1, 0.197, 0.378, 0.756],
})
df_test_TSCS_shift = pd.DataFrame({
    '1D_Trans_CC': [0.024, 0.015, 0.048, 0.153, 0.546],
    '1D_Trans_CC_sparse': [0.003, 0.017, 0.06, 0.141, 0.6],
    '1D_Trans_TT': [0.004, 0.007, 0.006, 0.074, 0.068],
    '1D_Trans_TT_sparse': [0.01, -0.0, -0.001, 0.034, 0.056],
    '2D_Trans_TC': [0.003, -0.004, 0.006, 0.007, 0.012],
    '2D_Trans_TCTC': [0.012, 0.011, 0.007, -0.014, 0.02],
    '2D_Trans_TCTC_sparse': [-0.002, 0.001, -0.011, -0.005, 0.013],
    '2D_Trans_TC_sparse': [0.007, -0.008, 0.005, -0.012, 0.015],
    'Global_MLP': [0.041, 0.059, 0.149, 0.305, 0.671],
    'theo_correl': [0.04, 0.1, 0.197, 0.378, 0.756],
})
df_test_TS_cond = pd.DataFrame({
    '1D_Trans_CC': [0.018, 0.016, 0.028, 0.073, 0.421],
    '1D_Trans_CC_sparse': [0.028, 0.022, 0.035, 0.082, 0.437],
    '1D_Trans_TT': [0.005, 0.009, 0.006, 0.002, 0.006],
    '1D_Trans_TT_sparse': [-0.002, -0.007, 0.006, -0.003, 0.003],
    '2D_Trans_TC': [0.012, 0.003, 0.031, -0.001, 0.307],
    '2D_Trans_TCTC': [-0.012, -0.0, 0.011, 0.004, 0.273],
    '2D_Trans_TCTC_sparse': [0.005, 0.015, 0.043, 0.031, 0.248],
    '2D_Trans_TC_sparse': [-0.006, -0.004, 0.023, 0.04, 0.267],
    'Global_MLP': [-0.001, 0.008, 0.012, 0.01, 0.018],
    'theo_correl': [0.04, 0.1, 0.197, 0.378, 0.756],
})
df_test_CS_cond = pd.DataFrame({
    '1D_Trans_CC': [-0.008, -0.009, -0.008, 0.004, 0.005],
    '1D_Trans_CC_sparse': [0.011, -0.009, 0.013, -0.005, 0.001],
    '1D_Trans_TT': [-0.005, -0.005, 0.007, 0.051, 0.113],
    '1D_Trans_TT_sparse': [0.007, 0.028, 0.015, 0.035, 0.118],
    '2D_Trans_TC': [-0.011, 0.002, 0.016, 0.072, 0.349],
    '2D_Trans_TCTC': [0.026, 0.025, 0.041, 0.028, 0.248],
    '2D_Trans_TCTC_sparse': [0.002, 0.03, 0.026, 0.078, 0.38],
    '2D_Trans_TC_sparse': [-0.001, 0.012, 0.022, 0.052, 0.339],
    'Global_MLP': [-0.011, -0.01, -0.001, -0.004, 0.01],
    'theo_correl': [0.04, 0.1, 0.197, 0.378, 0.756],
})

### Creating figures for the paper

## n_f = 20

In [ ]:
#all_effects_nf20_train_dic = {}
#all_effects_nf20_test_dic = {}

In [55]:
for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift', 'TS_cond', 'CS_cond']:
    print("Running_effect:", effect)
    out_train1, out_test1 = run_models1([0.02, 0.05, 0.1, 0.2, 0.5], effect, T=4000, n_ts=10, n_f=20, n_rolling=10)
    all_effects_nf20_train_dic[effect] = out_train1
    all_effects_nf20_test_dic[effect] = out_test1

Running_effect: lin
Running correl level: 0.02


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.05


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.1


...

/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.2


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.5


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [56]:
for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift', 'TS_cond', 'CS_cond']:
    print(effect)
    display(all_effects_nf20_train_dic[effect].T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
    display(all_effects_nf20_test_dic[effect].T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

lin


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
1D_Trans_CC,0.026,0.071,0.122,0.261,0.599
1D_Trans_CC_sparse,0.021,0.074,0.127,0.262,0.614
1D_Trans_TT,0.019,0.069,0.125,0.251,0.599
1D_Trans_TT_sparse,0.023,0.067,0.122,0.256,0.598
2D_Trans_TC,0.033,0.123,0.187,0.441,0.834
2D_Trans_TCTC,0.044,0.112,0.157,0.326,0.757
2D_Trans_TCTC_sparse,0.039,0.122,0.164,0.399,0.810
2D_Trans_TC_sparse,0.067,0.156,0.214,0.513,0.877
Global_MLP,0.018,0.055,0.096,0.203,0.496


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
1D_Trans_CC,0.020,0.068,0.147,0.321,0.684
1D_Trans_CC_sparse,0.030,0.085,0.148,0.314,0.673
1D_Trans_TT,0.026,0.071,0.130,0.257,0.592
1D_Trans_TT_sparse,0.031,0.065,0.124,0.269,0.599
2D_Trans_TC,0.032,0.129,0.201,0.442,0.836
2D_Trans_TCTC,0.055,0.135,0.185,0.348,0.773
2D_Trans_TCTC_sparse,0.053,0.145,0.196,0.423,0.816
2D_Trans_TC_sparse,0.066,0.143,0.210,0.523,0.878
Global_MLP,0.030,0.056,0.084,0.187,0.437


TS_shift


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
1D_Trans_CC,0.015,0.058,0.124,0.237,0.604
1D_Trans_CC_sparse,0.016,0.068,0.129,0.248,0.613
1D_Trans_TT,0.006,0.049,0.085,0.172,0.437
1D_Trans_TT_sparse,0.013,0.046,0.092,0.166,0.438
2D_Trans_TC,0.011,0.043,0.064,0.175,0.620
2D_Trans_TCTC,0.006,0.039,0.075,0.171,0.611
2D_Trans_TCTC_sparse,0.007,0.050,0.060,0.193,0.650
2D_Trans_TC_sparse,0.002,0.056,0.089,0.206,0.683
Global_MLP,0.013,0.048,0.098,0.187,0.494


...

rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
1D_Trans_CC,0.021,0.045,0.097,0.181,0.466
1D_Trans_CC_sparse,0.021,0.045,0.095,0.186,0.462
1D_Trans_TT,0.020,0.046,0.087,0.172,0.444
1D_Trans_TT_sparse,0.015,0.044,0.090,0.178,0.442
2D_Trans_TC,0.011,0.018,0.022,0.101,0.300
2D_Trans_TCTC,0.006,0.030,0.073,0.098,0.262
2D_Trans_TCTC_sparse,0.014,0.022,0.059,0.073,0.292
2D_Trans_TC_sparse,0.008,0.020,0.043,0.078,0.203
Global_MLP,0.022,0.049,0.101,0.198,0.495


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
1D_Trans_CC,0.018,0.006,0.004,0.013,0.065
1D_Trans_CC_sparse,-0.000,0.007,0.015,0.017,0.078
1D_Trans_TT,-0.010,-0.002,0.006,0.008,-0.003
1D_Trans_TT_sparse,-0.016,-0.003,-0.005,0.002,-0.004
2D_Trans_TC,0.003,0.004,-0.001,-0.006,0.049
2D_Trans_TCTC,-0.000,-0.003,0.018,-0.006,0.033
2D_Trans_TCTC_sparse,-0.011,-0.007,0.010,-0.003,0.038
2D_Trans_TC_sparse,-0.007,-0.011,0.009,-0.008,0.002
Global_MLP,-0.002,0.006,0.004,0.003,0.012


CS_cond


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
1D_Trans_CC,0.033,0.050,0.085,0.166,0.447
1D_Trans_CC_sparse,0.035,0.052,0.082,0.163,0.445
1D_Trans_TT,0.033,0.051,0.084,0.173,0.440
1D_Trans_TT_sparse,0.034,0.049,0.082,0.167,0.439
2D_Trans_TC,0.019,0.033,0.060,0.107,0.141
2D_Trans_TCTC,0.028,0.043,0.061,0.046,0.375
2D_Trans_TCTC_sparse,0.016,0.032,0.053,0.092,0.310
2D_Trans_TC_sparse,0.005,0.020,0.040,0.082,0.234
Global_MLP,0.036,0.053,0.093,0.184,0.485


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
1D_Trans_CC,-0.001,0.004,0.008,0.008,0.006
1D_Trans_CC_sparse,0.017,0.001,-0.005,0.004,-0.003
1D_Trans_TT,-0.001,-0.001,0.006,0.020,0.004
1D_Trans_TT_sparse,0.001,0.011,-0.008,0.001,0.006
2D_Trans_TC,-0.005,0.001,-0.003,0.009,0.012
2D_Trans_TCTC,-0.010,-0.007,0.005,0.005,0.030
2D_Trans_TCTC_sparse,0.007,0.004,0.005,0.010,0.046
2D_Trans_TC_sparse,-0.003,-0.004,-0.004,0.004,0.030
Global_MLP,0.004,-0.012,0.014,0.022,-0.007


### Hardcoded df

In [60]:
for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift', 'TS_cond', 'CS_cond']:
    print(generate_dataframe_code(all_effects_nf20_train_dic[effect], var_name="df_nf20_train_" + effect))


df_nf20_train_lin = pd.DataFrame({
    '1D_Trans_CC': [0.026, 0.071, 0.122, 0.261, 0.599],
    '1D_Trans_CC_sparse': [0.021, 0.074, 0.127, 0.262, 0.614],
    '1D_Trans_TT': [0.019, 0.069, 0.125, 0.251, 0.599],
    '1D_Trans_TT_sparse': [0.023, 0.067, 0.122, 0.256, 0.598],
    '2D_Trans_TC': [0.033, 0.123, 0.187, 0.441, 0.834],
    '2D_Trans_TCTC': [0.044, 0.112, 0.157, 0.326, 0.757],
    '2D_Trans_TCTC_sparse': [0.039, 0.122, 0.164, 0.399, 0.81],
    '2D_Trans_TC_sparse': [0.067, 0.156, 0.214, 0.513, 0.877],
    'Global_MLP': [0.018, 0.055, 0.096, 0.203, 0.496],
    'theo_correl': [0.022, 0.056, 0.112, 0.222, 0.542],
})
df_nf20_train_TS_shift = pd.DataFrame({
    '1D_Trans_CC': [0.015, 0.058, 0.124, 0.237, 0.604],
    '1D_Trans_CC_sparse': [0.016, 0.068, 0.129, 0.248, 0.613],
    '1D_Trans_TT': [0.006, 0.049, 0.085, 0.172, 0.437],
    '1D_Trans_TT_sparse': [0.013, 0.046, 0.092, 0.166, 0.438],
    '2D_Trans_TC': [0.011, 0.043, 0.064, 0.175, 0.62],
    '2D_Trans_TCTC': [0.006, 0.039, 0.0

In [61]:
df_nf20_train_lin = pd.DataFrame({
    '1D_Trans_CC': [0.026, 0.071, 0.122, 0.261, 0.599],
    '1D_Trans_CC_sparse': [0.021, 0.074, 0.127, 0.262, 0.614],
    '1D_Trans_TT': [0.019, 0.069, 0.125, 0.251, 0.599],
    '1D_Trans_TT_sparse': [0.023, 0.067, 0.122, 0.256, 0.598],
    '2D_Trans_TC': [0.033, 0.123, 0.187, 0.441, 0.834],
    '2D_Trans_TCTC': [0.044, 0.112, 0.157, 0.326, 0.757],
    '2D_Trans_TCTC_sparse': [0.039, 0.122, 0.164, 0.399, 0.81],
    '2D_Trans_TC_sparse': [0.067, 0.156, 0.214, 0.513, 0.877],
    'Global_MLP': [0.018, 0.055, 0.096, 0.203, 0.496],
    'theo_correl': [0.022, 0.056, 0.112, 0.222, 0.542],
})
df_nf20_train_TS_shift = pd.DataFrame({
    '1D_Trans_CC': [0.015, 0.058, 0.124, 0.237, 0.604],
    '1D_Trans_CC_sparse': [0.016, 0.068, 0.129, 0.248, 0.613],
    '1D_Trans_TT': [0.006, 0.049, 0.085, 0.172, 0.437],
    '1D_Trans_TT_sparse': [0.013, 0.046, 0.092, 0.166, 0.438],
    '2D_Trans_TC': [0.011, 0.043, 0.064, 0.175, 0.62],
    '2D_Trans_TCTC': [0.006, 0.039, 0.075, 0.171, 0.611],
    '2D_Trans_TCTC_sparse': [0.007, 0.05, 0.06, 0.193, 0.65],
    '2D_Trans_TC_sparse': [0.002, 0.056, 0.089, 0.206, 0.683],
    'Global_MLP': [0.013, 0.048, 0.098, 0.187, 0.494],
    'theo_correl': [0.022, 0.056, 0.112, 0.222, 0.542],
})
df_nf20_train_CS_shift = pd.DataFrame({
    '1D_Trans_CC': [0.017, 0.047, 0.098, 0.19, 0.483],
    '1D_Trans_CC_sparse': [0.014, 0.039, 0.101, 0.185, 0.49],
    '1D_Trans_TT': [0.018, 0.061, 0.136, 0.246, 0.601],
    '1D_Trans_TT_sparse': [0.023, 0.059, 0.137, 0.246, 0.602],
    '2D_Trans_TC': [0.026, 0.052, 0.142, 0.267, 0.687],
    '2D_Trans_TCTC': [0.016, 0.06, 0.143, 0.249, 0.651],
    '2D_Trans_TCTC_sparse': [0.011, 0.053, 0.164, 0.287, 0.689],
    '2D_Trans_TC_sparse': [0.022, 0.048, 0.161, 0.309, 0.7],
    'Global_MLP': [0.017, 0.045, 0.109, 0.192, 0.5],
    'theo_correl': [0.022, 0.056, 0.112, 0.222, 0.542],
})
df_nf20_train_fea_cond = pd.DataFrame({
    '1D_Trans_CC': [0.022, 0.045, 0.091, 0.188, 0.493],
    '1D_Trans_CC_sparse': [0.02, 0.05, 0.089, 0.19, 0.486],
    '1D_Trans_TT': [0.016, 0.048, 0.092, 0.181, 0.451],
    '1D_Trans_TT_sparse': [0.024, 0.044, 0.085, 0.185, 0.444],
    '2D_Trans_TC': [0.028, 0.072, 0.145, 0.335, 0.662],
    '2D_Trans_TCTC': [0.03, 0.083, 0.134, 0.286, 0.613],
    '2D_Trans_TCTC_sparse': [0.046, 0.094, 0.16, 0.308, 0.622],
    '2D_Trans_TC_sparse': [0.017, 0.089, 0.169, 0.342, 0.648],
    'Global_MLP': [0.021, 0.054, 0.096, 0.202, 0.497],
    'theo_correl': [0.022, 0.056, 0.112, 0.222, 0.542],
})
df_nf20_train_TSCS_shift = pd.DataFrame({
    '1D_Trans_CC': [0.021, 0.054, 0.1, 0.185, 0.475],
    '1D_Trans_CC_sparse': [0.02, 0.06, 0.098, 0.186, 0.489],
    '1D_Trans_TT': [0.018, 0.054, 0.099, 0.178, 0.442],
    '1D_Trans_TT_sparse': [0.019, 0.053, 0.097, 0.174, 0.442],
    '2D_Trans_TC': [0.01, 0.034, 0.047, 0.093, 0.263],
    '2D_Trans_TCTC': [0.012, 0.032, 0.057, 0.13, 0.314],
    '2D_Trans_TCTC_sparse': [0.016, 0.023, 0.058, 0.084, 0.251],
    '2D_Trans_TC_sparse': [0.006, 0.027, 0.039, 0.09, 0.224],
    'Global_MLP': [0.022, 0.057, 0.108, 0.196, 0.496],
    'theo_correl': [0.022, 0.056, 0.112, 0.222, 0.542],
})
df_nf20_train_TS_cond = pd.DataFrame({
    '1D_Trans_CC': [0.021, 0.045, 0.097, 0.181, 0.466],
    '1D_Trans_CC_sparse': [0.021, 0.045, 0.095, 0.186, 0.462],
    '1D_Trans_TT': [0.02, 0.046, 0.087, 0.172, 0.444],
    '1D_Trans_TT_sparse': [0.015, 0.044, 0.09, 0.178, 0.442],
    '2D_Trans_TC': [0.011, 0.018, 0.022, 0.101, 0.3],
    '2D_Trans_TCTC': [0.006, 0.03, 0.073, 0.098, 0.262],
    '2D_Trans_TCTC_sparse': [0.014, 0.022, 0.059, 0.073, 0.292],
    '2D_Trans_TC_sparse': [0.008, 0.02, 0.043, 0.078, 0.203],
    'Global_MLP': [0.022, 0.049, 0.101, 0.198, 0.495],
    'theo_correl': [0.022, 0.056, 0.112, 0.222, 0.542],
})
df_nf20_train_CS_cond = pd.DataFrame({
    '1D_Trans_CC': [0.033, 0.05, 0.085, 0.166, 0.447],
    '1D_Trans_CC_sparse': [0.035, 0.052, 0.082, 0.163, 0.445],
    '1D_Trans_TT': [0.033, 0.051, 0.084, 0.173, 0.44],
    '1D_Trans_TT_sparse': [0.034, 0.049, 0.082, 0.167, 0.439],
    '2D_Trans_TC': [0.019, 0.033, 0.06, 0.107, 0.141],
    '2D_Trans_TCTC': [0.028, 0.043, 0.061, 0.046, 0.375],
    '2D_Trans_TCTC_sparse': [0.016, 0.032, 0.053, 0.092, 0.31],
    '2D_Trans_TC_sparse': [0.005, 0.02, 0.04, 0.082, 0.234],
    'Global_MLP': [0.036, 0.053, 0.093, 0.184, 0.485],
    'theo_correl': [0.022, 0.056, 0.112, 0.222, 0.542],
})

In [ ]:
for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift', 'TS_cond', 'CS_cond']:
    print(generate_dataframe_code(all_effects_nf20_test_dic[effect], var_name="df_nf20_test_" + effect))


df_nf20_test_lin = pd.DataFrame({
    '1D_Trans_CC': [0.02, 0.068, 0.147, 0.321, 0.684],
    '1D_Trans_CC_sparse': [0.03, 0.085, 0.148, 0.314, 0.673],
    '1D_Trans_TT': [0.026, 0.071, 0.13, 0.257, 0.592],
    '1D_Trans_TT_sparse': [0.031, 0.065, 0.124, 0.269, 0.599],
    '2D_Trans_TC': [0.032, 0.129, 0.201, 0.442, 0.836],
    '2D_Trans_TCTC': [0.055, 0.135, 0.185, 0.348, 0.773],
    '2D_Trans_TCTC_sparse': [0.053, 0.145, 0.196, 0.423, 0.816],
    '2D_Trans_TC_sparse': [0.066, 0.143, 0.21, 0.523, 0.878],
    'Global_MLP': [0.03, 0.056, 0.084, 0.187, 0.437],
    'theo_correl': [0.01, 0.025, 0.05, 0.102, 0.277],
})
df_nf20_test_TS_shift = pd.DataFrame({
    '1D_Trans_CC': [0.02, 0.056, 0.144, 0.286, 0.676],
    '1D_Trans_CC_sparse': [0.037, 0.071, 0.153, 0.282, 0.69],
    '1D_Trans_TT': [0.002, -0.002, 0.014, 0.005, 0.015],
    '1D_Trans_TT_sparse': [0.013, -0.01, 0.006, -0.005, 0.001],
    '2D_Trans_TC': [-0.006, 0.011, 0.023, 0.091, 0.524],
    '2D_Trans_TCTC': [-0.002, 0.008, 0.018, 0

In [59]:
df_nf20_test_lin = pd.DataFrame({
    '1D_Trans_CC': [0.02, 0.068, 0.147, 0.321, 0.684],
    '1D_Trans_CC_sparse': [0.03, 0.085, 0.148, 0.314, 0.673],
    '1D_Trans_TT': [0.026, 0.071, 0.13, 0.257, 0.592],
    '1D_Trans_TT_sparse': [0.031, 0.065, 0.124, 0.269, 0.599],
    '2D_Trans_TC': [0.032, 0.129, 0.201, 0.442, 0.836],
    '2D_Trans_TCTC': [0.055, 0.135, 0.185, 0.348, 0.773],
    '2D_Trans_TCTC_sparse': [0.053, 0.145, 0.196, 0.423, 0.816],
    '2D_Trans_TC_sparse': [0.066, 0.143, 0.21, 0.523, 0.878],
    'Global_MLP': [0.03, 0.056, 0.084, 0.187, 0.437],
    'theo_correl': [0.01, 0.025, 0.05, 0.102, 0.277],
})
df_nf20_test_TS_shift = pd.DataFrame({
    '1D_Trans_CC': [0.02, 0.056, 0.144, 0.286, 0.676],
    '1D_Trans_CC_sparse': [0.037, 0.071, 0.153, 0.282, 0.69],
    '1D_Trans_TT': [0.002, -0.002, 0.014, 0.005, 0.015],
    '1D_Trans_TT_sparse': [0.013, -0.01, 0.006, -0.005, 0.001],
    '2D_Trans_TC': [-0.006, 0.011, 0.023, 0.091, 0.524],
    '2D_Trans_TCTC': [-0.002, 0.008, 0.018, 0.063, 0.491],
    '2D_Trans_TCTC_sparse': [0.007, 0.034, 0.041, 0.109, 0.568],
    '2D_Trans_TC_sparse': [0.007, 0.032, 0.062, 0.133, 0.632],
    'Global_MLP': [0.012, 0.038, 0.101, 0.156, 0.443],
    'theo_correl': [0.01, 0.025, 0.05, 0.102, 0.277],
})
df_nf20_test_CS_shift = pd.DataFrame({
    '1D_Trans_CC': [-0.004, 0.014, 0.015, 0.029, 0.143],
    '1D_Trans_CC_sparse': [0.006, 0.018, 0.005, 0.036, 0.188],
    '1D_Trans_TT': [0.028, 0.047, 0.134, 0.246, 0.598],
    '1D_Trans_TT_sparse': [0.016, 0.064, 0.135, 0.23, 0.604],
    '2D_Trans_TC': [0.02, 0.034, 0.107, 0.226, 0.652],
    '2D_Trans_TCTC': [0.015, 0.035, 0.107, 0.229, 0.617],
    '2D_Trans_TCTC_sparse': [0.003, 0.049, 0.146, 0.263, 0.67],
    '2D_Trans_TC_sparse': [0.019, 0.032, 0.135, 0.272, 0.677],
    'Global_MLP': [0.011, 0.05, 0.097, 0.161, 0.46],
    'theo_correl': [0.01, 0.025, 0.05, 0.102, 0.277],
})
df_nf20_test_fea_cond = pd.DataFrame({
    '1D_Trans_CC': [0.004, -0.014, 0.012, 0.01, 0.168],
    '1D_Trans_CC_sparse': [0.008, 0.007, -0.019, 0.018, 0.166],
    '1D_Trans_TT': [-0.014, -0.003, 0.009, 0.005, 0.013],
    '1D_Trans_TT_sparse': [0.007, -0.0, 0.004, 0.012, 0.009],
    '2D_Trans_TC': [0.017, 0.056, 0.13, 0.295, 0.599],
    '2D_Trans_TCTC': [0.03, 0.071, 0.115, 0.265, 0.548],
    '2D_Trans_TCTC_sparse': [0.048, 0.057, 0.143, 0.278, 0.556],
    '2D_Trans_TC_sparse': [0.007, 0.062, 0.153, 0.304, 0.581],
    'Global_MLP': [-0.019, -0.007, -0.005, -0.009, -0.001],
    'theo_correl': [0.01, 0.025, 0.05, 0.102, 0.277],
})
df_nf20_test_TSCS_shift = pd.DataFrame({
    '1D_Trans_CC': [0.001, 0.015, 0.019, 0.04, 0.158],
    '1D_Trans_CC_sparse': [-0.003, 0.0, 0.008, 0.028, 0.184],
    '1D_Trans_TT': [-0.006, 0.006, -0.009, 0.003, 0.009],
    '1D_Trans_TT_sparse': [0.008, -0.0, -0.007, 0.007, -0.0],
    '2D_Trans_TC': [0.004, -0.007, -0.014, 0.003, -0.0],
    '2D_Trans_TCTC': [-0.007, 0.005, -0.003, -0.01, -0.0],
    '2D_Trans_TCTC_sparse': [-0.003, -0.001, -0.0, 0.011, 0.001],
    '2D_Trans_TC_sparse': [0.008, 0.004, 0.003, -0.006, -0.01],
    'Global_MLP': [0.031, 0.048, 0.089, 0.177, 0.442],
    'theo_correl': [0.01, 0.025, 0.05, 0.102, 0.277],
})
df_nf20_test_TS_cond = pd.DataFrame({
    '1D_Trans_CC': [0.018, 0.006, 0.004, 0.013, 0.065],
    '1D_Trans_CC_sparse': [-0.0, 0.007, 0.015, 0.017, 0.078],
    '1D_Trans_TT': [-0.01, -0.002, 0.006, 0.008, -0.003],
    '1D_Trans_TT_sparse': [-0.016, -0.003, -0.005, 0.002, -0.004],
    '2D_Trans_TC': [0.003, 0.004, -0.001, -0.006, 0.049],
    '2D_Trans_TCTC': [-0.0, -0.003, 0.018, -0.006, 0.033],
    '2D_Trans_TCTC_sparse': [-0.011, -0.007, 0.01, -0.003, 0.038],
    '2D_Trans_TC_sparse': [-0.007, -0.011, 0.009, -0.008, 0.002],
    'Global_MLP': [-0.002, 0.006, 0.004, 0.003, 0.012],
    'theo_correl': [0.01, 0.025, 0.05, 0.102, 0.277],
})
df_nf20_test_CS_cond = pd.DataFrame({
    '1D_Trans_CC': [-0.001, 0.004, 0.008, 0.008, 0.006],
    '1D_Trans_CC_sparse': [0.017, 0.001, -0.005, 0.004, -0.003],
    '1D_Trans_TT': [-0.001, -0.001, 0.006, 0.02, 0.004],
    '1D_Trans_TT_sparse': [0.001, 0.011, -0.008, 0.001, 0.006],
    '2D_Trans_TC': [-0.005, 0.001, -0.003, 0.009, 0.012],
    '2D_Trans_TCTC': [-0.01, -0.007, 0.005, 0.005, 0.03],
    '2D_Trans_TCTC_sparse': [0.007, 0.004, 0.005, 0.01, 0.046],
    '2D_Trans_TC_sparse': [-0.003, -0.004, -0.004, 0.004, 0.03],
    'Global_MLP': [0.004, -0.012, 0.014, 0.022, -0.007],
    'theo_correl': [0.01, 0.025, 0.05, 0.102, 0.277],
})

# Run of models on sum of effects

In [26]:
# In this section let's run all of our models on the superpositions of all the effects.

In [29]:
def run_sum_models1(rhos, T=4000, n_ts=10, n_rolling=10):

    list_effects1 = ["lin", "TS_shift", "CS_shift", "TSCS_shift", "fea_cond", "TS_cond", "CS_cond"]
    n_f = 7
    correl_split_by_fea1 = [1/np.sqrt(n_f)]*n_f

    mask = causal_mask
    mask_c = build_attention_mask(mask, n_rolling, device=device)
    
    z = generators.Generator(T, n_ts, n_f)
    
    train_dic = {}
    test_dic = {}

    for (i,rho) in enumerate(rhos):
        print("Running correl level:", rho)
        theo_correl_is = get_ols_corr(n_ts*n_f*n_rolling, T*0.625, rho, oos=False)
        theo_correl_oos = get_ols_corr(n_ts*n_f*n_rolling, T*0.625, rho, oos=True)


        z.generate_dataset_gr_simple(
            global_corr=rho, 
            correl_split_by_fea=correl_split_by_fea1, 
            list_type_effects=list_effects1,
            list_type_interaction=["cond"], 
            random_ts_shift=n_rolling,
        )


        models_torch = {
        'Global_MLP': models.GlobalMLP(n_ts, n_f, n_rolling, dropout=0.1).to(device),
        '1D_Trans_TT': models.OneDimensionalTransformer(n_ts, n_f, n_rolling, mask=mask_c, attn_direction="T",  num_attn_layers=2,
                                        d_model=64, dim_feedforward=256, nhead=8, compression="MLP", num_mlp_layers=2,
                                        dropout=0.1, roll_y=True).to(device),
        '1D_Trans_CC': models.OneDimensionalTransformer(n_ts, n_f, n_rolling, mask=mask_c, attn_direction="C",  num_attn_layers=2,
                                        d_model=64, dim_feedforward=256, nhead=8, compression="MLP", num_mlp_layers=2,
                                        dropout=0.1,  roll_y=False).to(device),                                
        '2D_Trans_TC': models.CustomBiDimensionalTransformer(
            n_ts, n_f, n_rolling, mask=mask_c, layers='TC', nhead=8,
            dropout=0.1, d_model=64, dim_feedforward=256, sparsify=None, roll_y=True, embeddings="both",
        ).to(device),
        '2D_Trans_TCTC': models.CustomBiDimensionalTransformer(
            n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
            dropout=0.1, d_model=64, dim_feedforward=256, sparsify=None, roll_y=True, embeddings="both",
        ).to(device),
        '1D_Trans_TT_sparse': models.OneDimensionalTransformer(n_ts, n_f, n_rolling, mask=mask_c, attn_direction="T",  num_attn_layers=2,
                                        d_model=64, dim_feedforward=256, nhead=8, compression="MLP", num_mlp_layers=2,
                                        dropout=0.1, roll_y=True, sparsify=keep_topk_per_row3,).to(device),
        '1D_Trans_CC_sparse': models.OneDimensionalTransformer(n_ts, n_f, n_rolling, mask=mask_c, attn_direction="C",  num_attn_layers=2,
                                        d_model=64, dim_feedforward=256, nhead=8, compression="MLP", num_mlp_layers=2,
                                        dropout=0.1,  roll_y=False, sparsify=keep_topk_per_row3,).to(device),                                
        '2D_Trans_TC_sparse': models.CustomBiDimensionalTransformer(
            n_ts, n_f, n_rolling, mask=mask_c, layers='TC', nhead=8,
            dropout=0.1, d_model=64, dim_feedforward=256, sparsify=keep_topk_per_row3, roll_y=True, embeddings="both",
        ).to(device),
        '2D_Trans_TCTC_sparse': models.CustomBiDimensionalTransformer(
            n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
            dropout=0.1, d_model=64, dim_feedforward=256, sparsify=keep_topk_per_row3, roll_y=True, embeddings="both",
        ).to(device),
        }

        models_torch = {
            k: torch_benchmarks.TorchWrapper(
                models_torch[k], 
                optimizer=torch.optim.AdamW(models_torch[k].parameters(), lr=0.001),
                loss_fn=nn.MSELoss()  
            ) for k in models_torch
        }

        
        for k in models_torch:
            roll_y = True
            if k in ['Global_MLP', '1D_Trans_CC', '1D_Trans_CC_sparse']:
                roll_y = False
            z.get_dataloader(n_rolling=n_rolling, roll_y=roll_y)
            epochs=40
            if k in ['1D_Trans_CC', '1D_Trans_CC_sparse']:
                epochs=80
            models_torch[k].fit(z.train, test=z.test, epochs=epochs, plot=False, verbose=0)
            
      
        comp = benchmark_comparison.Comparator(models=[models_torch[k] for k in models_torch], 
        model_names=[k for k in models_torch]
        )

        train_dic[rho] = comp.correl(z, mode="train", return_values=True)
        test_dic[rho]  = comp.correl(z, mode="test",  return_values=True)

        
    return train_dic, test_dic

## n_f = 7

In [40]:
dic_sum_effects_train1, dic_sum_effects_test1 = run_sum_models1([0.02, 0.05, 0.1, 0.2, 0.5])

Running correl level: 0.02


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.05


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.1


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.2


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.5


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [41]:
def display_table1(df):
    df_display = df.loc["Global_MLP":]
    df_display = df_display.loc[:, "optimal":"conditional_cs"]
    display(df_display.style.background_gradient(cmap='coolwarm', axis=0, vmin=None, vmax=None).format(precision=3))

In [42]:
for rho in [0.02, 0.05, 0.1, 0.2, 0.5]:
    print(rho)
    display_table1(dic_sum_effects_test1[rho])

0.02


,optimal,linear,conditional,shift,cs,cs_shift,conditional_ts,conditional_cs
Global_MLP,0.020,0.017,-0.004,0.021,0.017,0.006,-0.010,0.004
1D_Trans_TT,0.014,0.014,-0.003,0.002,0.030,-0.002,-0.015,0.010
1D_Trans_CC,0.004,0.010,-0.003,0.005,-0.001,-0.002,0.000,0.003
2D_Trans_TC,0.026,0.029,0.023,0.007,0.010,0.001,0.009,-0.010
2D_Trans_TCTC,0.004,-0.021,0.017,0.008,0.017,-0.009,-0.007,0.004
1D_Trans_TT_sparse,-0.001,0.008,-0.014,0.004,0.013,-0.012,-0.008,0.006
1D_Trans_CC_sparse,0.014,0.019,-0.002,-0.005,0.002,0.005,0.017,0.002
2D_Trans_TC_sparse,0.037,0.031,0.016,0.002,0.040,0.000,-0.002,0.008
2D_Trans_TCTC_sparse,0.018,0.027,0.020,-0.016,0.013,-0.001,-0.005,0.008


0.05


,optimal,linear,conditional,shift,cs,cs_shift,conditional_ts,conditional_cs
Global_MLP,0.047,0.050,-0.006,0.024,0.035,0.034,-0.006,-0.007
1D_Trans_TT,0.026,0.012,0.003,-0.008,0.047,0.011,-0.001,0.006
1D_Trans_CC,0.037,0.053,-0.011,0.016,0.020,0.019,-0.000,0.001
2D_Trans_TC,0.056,0.037,0.053,0.016,0.048,0.006,-0.015,0.003
2D_Trans_TCTC,0.105,0.109,0.043,0.015,0.109,0.006,-0.002,-0.000
1D_Trans_TT_sparse,0.059,0.038,0.006,0.014,0.054,0.009,0.019,0.016
1D_Trans_CC_sparse,0.035,0.039,0.002,0.038,0.018,-0.004,-0.001,0.001
2D_Trans_TC_sparse,0.081,0.076,0.058,0.014,0.057,0.006,0.003,0.002
2D_Trans_TCTC_sparse,0.101,0.128,0.071,0.004,0.051,0.008,-0.003,0.011


0.1


,optimal,linear,conditional,shift,cs,cs_shift,conditional_ts,conditional_cs
Global_MLP,0.104,0.060,0.008,0.061,0.087,0.068,0.007,-0.017
1D_Trans_TT,0.094,0.083,0.006,-0.007,0.116,0.015,0.023,0.011
1D_Trans_CC,0.072,0.082,0.023,0.062,0.020,0.026,-0.005,-0.018
2D_Trans_TC,0.147,0.112,0.108,0.023,0.150,0.003,-0.011,0.005
2D_Trans_TCTC,0.131,0.090,0.111,0.014,0.129,0.004,0.009,-0.012
1D_Trans_TT_sparse,0.064,0.059,0.000,0.008,0.083,0.009,-0.002,0.011
1D_Trans_CC_sparse,0.105,0.069,0.032,0.089,0.047,0.031,0.012,-0.004
2D_Trans_TC_sparse,0.219,0.205,0.154,0.021,0.212,0.001,-0.003,-0.011
2D_Trans_TCTC_sparse,0.130,0.071,0.087,0.036,0.131,0.003,0.010,0.004


0.2


,optimal,linear,conditional,shift,cs,cs_shift,conditional_ts,conditional_cs
Global_MLP,0.145,0.093,-0.002,0.101,0.101,0.114,-0.014,-0.011
1D_Trans_TT,0.108,0.134,0.011,0.005,0.124,0.007,-0.011,0.014
1D_Trans_CC,0.123,0.113,0.012,0.125,0.028,0.019,0.023,0.004
2D_Trans_TC,0.196,0.172,0.174,-0.005,0.163,-0.003,0.006,0.011
2D_Trans_TCTC,0.190,0.174,0.155,0.015,0.146,0.003,0.009,0.003
1D_Trans_TT_sparse,0.090,0.110,0.010,-0.011,0.124,-0.004,-0.002,0.013
1D_Trans_CC_sparse,0.120,0.104,0.025,0.118,0.029,0.029,0.010,0.001
2D_Trans_TC_sparse,0.262,0.243,0.186,0.027,0.173,0.000,0.043,0.023
2D_Trans_TCTC_sparse,0.215,0.199,0.189,0.013,0.159,-0.006,0.002,0.015


0.5


,optimal,linear,conditional,shift,cs,cs_shift,conditional_ts,conditional_cs
Global_MLP,0.390,0.252,0.009,0.254,0.260,0.251,0.001,0.014
1D_Trans_TT,0.258,0.294,0.032,0.002,0.329,0.006,0.005,0.021
1D_Trans_CC,0.476,0.289,0.169,0.287,0.164,0.197,0.154,0.008
2D_Trans_TC,0.634,0.382,0.318,0.323,0.375,0.007,0.240,0.048
2D_Trans_TCTC,0.593,0.363,0.313,0.287,0.362,0.002,0.219,0.036
1D_Trans_TT_sparse,0.250,0.302,0.022,0.008,0.295,0.007,0.002,0.031
1D_Trans_CC_sparse,0.506,0.290,0.176,0.295,0.209,0.208,0.170,0.003
2D_Trans_TC_sparse,0.649,0.395,0.334,0.320,0.368,0.001,0.200,0.115
2D_Trans_TCTC_sparse,0.606,0.368,0.316,0.287,0.353,-0.001,0.234,0.059


In [43]:
# Ok great!!

### Hardcoded df

In [49]:
for rho in [0.02, 0.05, 0.1, 0.2, 0.5]:
    print(generate_dataframe_code(dic_sum_effects_train1[rho], var_name="df_train_all_effects_rho" + str(rho)[2:]))

df_train_all_effects_rho02 = pd.DataFrame({
    'true': [0.022, 0.013, 0.013, 0.003, 0.005, 0.006, 0.011, 0.008, 0.992, 0.797, 0.856, 0.422, 0.481, 0.798, 0.832, 0.217, 0.279],
    'optimal': [np.nan, 0.37, 0.381, 0.374, 0.387, 0.379, 0.381, 0.373, 0.02, 0.026, 0.025, 0.045, 0.021, 0.021, 0.017, 0.052, 0.034],
    'linear': [np.nan, np.nan, -0.011, -0.011, 0.013, -0.002, 0.004, -0.005, 0.012, 0.018, 0.019, 0.046, 0.007, 0.017, 0.019, 0.053, 0.033],
    'conditional': [np.nan, np.nan, np.nan, 0.009, 0.008, -0.008, 0.007, -0.007, 0.013, 0.009, 0.012, 0.034, 0.015, 0.008, 0.005, 0.022, 0.051],
    'shift': [np.nan, np.nan, np.nan, np.nan, 0.005, 0.002, -0.011, -0.002, 0.003, 0.001, 0.005, 0.003, 0.011, 0.001, 0.001, 0.004, -0.02],
    'cs': [np.nan, np.nan, np.nan, np.nan, np.nan, 0.005, -0.008, 0.005, 0.002, 0.016, -0.002, 0.022, 0.006, 0.005, -0.0, 0.034, 0.022],
    'cs_shift': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.014, -0.009, 0.005, 0.009, 0.006, 0.002, 0.001, -0.001, 0.

In [51]:
df_train_all_effects_rho02 = pd.DataFrame({
    'true': [0.022, 0.013, 0.013, 0.003, 0.005, 0.006, 0.011, 0.008, 0.992, 0.797, 0.856, 0.422, 0.481, 0.798, 0.832, 0.217, 0.279],
    'optimal': [np.nan, 0.37, 0.381, 0.374, 0.387, 0.379, 0.381, 0.373, 0.02, 0.026, 0.025, 0.045, 0.021, 0.021, 0.017, 0.052, 0.034],
    'linear': [np.nan, np.nan, -0.011, -0.011, 0.013, -0.002, 0.004, -0.005, 0.012, 0.018, 0.019, 0.046, 0.007, 0.017, 0.019, 0.053, 0.033],
    'conditional': [np.nan, np.nan, np.nan, 0.009, 0.008, -0.008, 0.007, -0.007, 0.013, 0.009, 0.012, 0.034, 0.015, 0.008, 0.005, 0.022, 0.051],
    'shift': [np.nan, np.nan, np.nan, np.nan, 0.005, 0.002, -0.011, -0.002, 0.003, 0.001, 0.005, 0.003, 0.011, 0.001, 0.001, 0.004, -0.02],
    'cs': [np.nan, np.nan, np.nan, np.nan, np.nan, 0.005, -0.008, 0.005, 0.002, 0.016, -0.002, 0.022, 0.006, 0.005, -0.0, 0.034, 0.022],
    'cs_shift': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.014, -0.009, 0.005, 0.009, 0.006, 0.002, 0.001, -0.001, 0.011, 0.0, 0.0],
    'conditional_ts': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.001, 0.011, 0.007, 0.019, 0.004, 0.002, 0.016, 0.009, 0.016, 0.006],
    'conditional_cs': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.008, 0.008, 0.009, 0.009, 0.014, 0.01, 0.001, 0.008, -0.003],
    'Global_MLP': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.792, 0.851, 0.422, 0.478, 0.792, 0.826, 0.217, 0.279],
    '1D_Trans_TT': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.689, 0.387, 0.43, 0.689, 0.67, 0.231, 0.283],
    '1D_Trans_CC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.377, 0.429, 0.686, 0.731, 0.205, 0.258],
    '2D_Trans_TC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.383, 0.383, 0.363, 0.306, 0.318],
    '2D_Trans_TCTC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.429, 0.41, 0.293, 0.318],
    '1D_Trans_TT_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.669, 0.229, 0.287],
    '1D_Trans_CC_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.201, 0.249],
    '2D_Trans_TC_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.321],
})
df_train_all_effects_rho05 = pd.DataFrame({
    'true': [0.057, 0.024, 0.021, 0.017, 0.026, 0.017, 0.023, 0.023, 0.992, 0.8, 0.851, 0.424, 0.276, 0.785, 0.825, 0.242, 0.279],
    'optimal': [np.nan, 0.373, 0.377, 0.389, 0.381, 0.385, 0.375, 0.374, 0.056, 0.063, 0.057, 0.071, 0.095, 0.06, 0.063, 0.084, 0.099],
    'linear': [np.nan, np.nan, -0.009, 0.007, -0.003, 0.001, -0.006, -0.003, 0.025, 0.034, 0.038, 0.044, 0.1, 0.044, 0.032, 0.085, 0.122],
    'conditional': [np.nan, np.nan, np.nan, 0.003, 0.009, 0.006, -0.003, -0.0, 0.021, 0.022, 0.019, 0.055, 0.044, 0.017, 0.02, 0.061, 0.06],
    'shift': [np.nan, np.nan, np.nan, np.nan, 0.004, 0.006, 0.011, 0.004, 0.018, 0.012, 0.014, 0.002, -0.001, 0.014, 0.03, 0.008, 0.011],
    'cs': [np.nan, np.nan, np.nan, np.nan, np.nan, 0.013, -0.0, -0.008, 0.024, 0.042, 0.026, 0.048, 0.095, 0.037, 0.032, 0.054, 0.057],
    'cs_shift': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, -0.004, 0.004, 0.016, 0.019, 0.017, 0.015, 0.004, 0.015, 0.019, -0.0, 0.01],
    'conditional_ts': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, -0.008, 0.022, 0.018, 0.02, 0.009, 0.002, 0.014, 0.017, 0.005, 0.005],
    'conditional_cs': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.023, 0.022, 0.019, 0.017, 0.007, 0.018, 0.02, 0.01, -0.002],
    'Global_MLP': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.796, 0.847, 0.422, 0.275, 0.781, 0.82, 0.241, 0.278],
    '1D_Trans_TT': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.687, 0.391, 0.284, 0.685, 0.67, 0.259, 0.28],
    '1D_Trans_CC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.371, 0.247, 0.674, 0.718, 0.218, 0.255],
    '2D_Trans_TC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.322, 0.382, 0.368, 0.295, 0.291],
    '2D_Trans_TCTC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.276, 0.25, 0.315, 0.319],
    '1D_Trans_TT_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.651, 0.245, 0.289],
    '1D_Trans_CC_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.225, 0.239],
    '2D_Trans_TC_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.31],
})
df_train_all_effects_rho1 = pd.DataFrame({
    'true': [0.118, 0.038, 0.043, 0.05, 0.057, 0.045, 0.031, 0.046, 0.992, 0.799, 0.855, 0.384, 0.469, 0.787, 0.83, 0.212, 0.436],
    'optimal': [np.nan, 0.377, 0.375, 0.377, 0.384, 0.378, 0.376, 0.377, 0.12, 0.123, 0.13, 0.163, 0.148, 0.126, 0.132, 0.237, 0.167],
    'linear': [np.nan, np.nan, -0.008, -0.002, 0.004, -0.002, -0.001, 0.002, 0.041, 0.058, 0.064, 0.108, 0.087, 0.055, 0.057, 0.193, 0.067],
    'conditional': [np.nan, np.nan, np.nan, -0.002, -0.002, -0.004, 0.006, -0.005, 0.044, 0.035, 0.038, 0.101, 0.103, 0.041, 0.05, 0.142, 0.091],
    'shift': [np.nan, np.nan, np.nan, np.nan, 0.01, 0.002, -0.012, 0.0, 0.05, 0.042, 0.07, 0.014, 0.032, 0.041, 0.072, 0.049, 0.062],
    'cs': [np.nan, np.nan, np.nan, np.nan, np.nan, 0.007, 0.0, -0.003, 0.059, 0.088, 0.053, 0.136, 0.112, 0.086, 0.061, 0.211, 0.135],
    'cs_shift': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.0, -0.004, 0.046, 0.037, 0.051, 0.025, 0.029, 0.038, 0.041, 0.008, 0.029],
    'conditional_ts': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.01, 0.032, 0.024, 0.028, 0.019, 0.015, 0.03, 0.031, 0.01, 0.02],
    'conditional_cs': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.046, 0.04, 0.04, 0.027, 0.012, 0.041, 0.038, 0.013, 0.037],
    'Global_MLP': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.794, 0.85, 0.383, 0.465, 0.784, 0.824, 0.212, 0.436],
    '1D_Trans_TT': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.692, 0.363, 0.428, 0.68, 0.672, 0.224, 0.408],
    '1D_Trans_CC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.343, 0.417, 0.683, 0.731, 0.202, 0.387],
    '2D_Trans_TC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.365, 0.367, 0.342, 0.331, 0.35],
    '2D_Trans_TCTC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.426, 0.411, 0.308, 0.379],
    '1D_Trans_TT_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.662, 0.236, 0.406],
    '1D_Trans_CC_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.209, 0.379],
    '2D_Trans_TC_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.302],
})
df_train_all_effects_rho2 = pd.DataFrame({
    'true': [0.191, 0.072, 0.074, 0.064, 0.068, 0.08, 0.077, 0.07, 0.992, 0.802, 0.853, 0.435, 0.493, 0.784, 0.821, 0.345, 0.394],
    'optimal': [np.nan, 0.386, 0.378, 0.372, 0.372, 0.382, 0.375, 0.378, 0.192, 0.2, 0.207, 0.259, 0.247, 0.197, 0.208, 0.295, 0.254],
    'linear': [np.nan, np.nan, 0.007, -0.009, 0.005, 0.01, 0.007, -0.003, 0.073, 0.114, 0.098, 0.186, 0.171, 0.113, 0.1, 0.249, 0.197],
    'conditional': [np.nan, np.nan, np.nan, 0.003, -0.008, 0.001, -0.003, 0.004, 0.073, 0.065, 0.073, 0.184, 0.164, 0.059, 0.075, 0.194, 0.192],
    'shift': [np.nan, np.nan, np.nan, np.nan, 0.003, 0.0, -0.007, -0.004, 0.065, 0.057, 0.094, 0.031, 0.037, 0.049, 0.09, 0.035, 0.032],
    'cs': [np.nan, np.nan, np.nan, np.nan, np.nan, -0.006, -0.008, -0.006, 0.07, 0.109, 0.07, 0.178, 0.158, 0.116, 0.063, 0.173, 0.166],
    'cs_shift': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.001, 0.001, 0.081, 0.062, 0.08, 0.031, 0.037, 0.065, 0.089, 0.025, 0.021],
    'conditional_ts': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.007, 0.077, 0.06, 0.076, 0.04, 0.047, 0.061, 0.077, 0.059, 0.03],
    'conditional_cs': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.068, 0.062, 0.057, 0.032, 0.038, 0.058, 0.056, 0.044, 0.032],
    'Global_MLP': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.796, 0.847, 0.434, 0.49, 0.779, 0.816, 0.344, 0.391],
    '1D_Trans_TT': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.696, 0.408, 0.449, 0.682, 0.669, 0.339, 0.386],
    '1D_Trans_CC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.392, 0.439, 0.679, 0.724, 0.318, 0.353],
    '2D_Trans_TC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.439, 0.397, 0.387, 0.408, 0.411],
    '2D_Trans_TCTC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.453, 0.429, 0.402, 0.424],
    '1D_Trans_TT_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.654, 0.345, 0.38],
    '1D_Trans_CC_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.319, 0.353],
    '2D_Trans_TC_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.405],
})
df_train_all_effects_rho5 = pd.DataFrame({
    'true': [0.51, 0.19, 0.189, 0.196, 0.189, 0.198, 0.197, 0.189, 0.993, 0.8, 0.86, 0.553, 0.621, 0.791, 0.842, 0.505, 0.564],
    'optimal': [np.nan, 0.383, 0.372, 0.38, 0.379, 0.382, 0.372, 0.376, 0.512, 0.496, 0.573, 0.689, 0.664, 0.505, 0.588, 0.695, 0.665],
    'linear': [np.nan, np.nan, -0.007, 0.007, 0.007, 0.007, -0.006, 0.005, 0.192, 0.288, 0.249, 0.391, 0.354, 0.295, 0.256, 0.392, 0.369],
    'conditional': [np.nan, np.nan, np.nan, -0.004, 0.004, 0.002, -0.012, -0.007, 0.188, 0.149, 0.214, 0.312, 0.314, 0.155, 0.208, 0.34, 0.317],
    'shift': [np.nan, np.nan, np.nan, np.nan, -0.0, -0.005, 0.003, 0.005, 0.199, 0.151, 0.259, 0.341, 0.31, 0.148, 0.269, 0.338, 0.317],
    'cs': [np.nan, np.nan, np.nan, np.nan, np.nan, 0.005, -0.001, -0.002, 0.194, 0.291, 0.211, 0.362, 0.344, 0.288, 0.223, 0.359, 0.337],
    'cs_shift': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.002, -0.006, 0.199, 0.146, 0.219, 0.058, 0.081, 0.148, 0.237, 0.041, 0.063],
    'conditional_ts': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, -0.004, 0.194, 0.141, 0.216, 0.258, 0.243, 0.144, 0.217, 0.219, 0.241],
    'conditional_cs': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.188, 0.147, 0.145, 0.099, 0.111, 0.158, 0.143, 0.151, 0.113],
    'Global_MLP': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.797, 0.856, 0.555, 0.621, 0.788, 0.839, 0.507, 0.565],
    '1D_Trans_TT': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.716, 0.548, 0.595, 0.73, 0.704, 0.511, 0.556],
    '1D_Trans_CC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.594, 0.633, 0.709, 0.786, 0.559, 0.595],
    '2D_Trans_TC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.735, 0.55, 0.598, 0.748, 0.732],
    '2D_Trans_TCTC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.595, 0.639, 0.714, 0.72],
    '1D_Trans_TT_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.7, 0.515, 0.558],
    '1D_Trans_CC_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.563, 0.602],
    '2D_Trans_TC_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.728],
})

In [52]:
for rho in [0.02, 0.05, 0.1, 0.2, 0.5]:
    print(generate_dataframe_code(dic_sum_effects_test1[rho], var_name="df_test_all_effects_rho" + str(rho)[2:]))

df_test_all_effects_rho02 = pd.DataFrame({
    'true': [0.024, 0.001, 0.006, -0.0, 0.013, 0.016, 0.02, 0.007, 0.008, -0.002, -0.004, 0.004, -0.001, -0.012, -0.006, 0.005, 0.003],
    'optimal': [np.nan, 0.384, 0.376, 0.387, 0.373, 0.377, 0.364, 0.375, 0.02, 0.014, 0.004, 0.026, 0.004, -0.001, 0.014, 0.037, 0.018],
    'linear': [np.nan, np.nan, 0.003, -0.003, 0.011, -0.001, -0.0, 0.0, 0.017, 0.014, 0.01, 0.029, -0.021, 0.008, 0.019, 0.031, 0.027],
    'conditional': [np.nan, np.nan, np.nan, 0.005, -0.012, 0.003, -0.003, -0.002, -0.004, -0.003, -0.003, 0.023, 0.017, -0.014, -0.002, 0.016, 0.02],
    'shift': [np.nan, np.nan, np.nan, np.nan, 0.001, 0.005, -0.0, 0.012, 0.021, 0.002, 0.005, 0.007, 0.008, 0.004, -0.005, 0.002, -0.016],
    'cs': [np.nan, np.nan, np.nan, np.nan, np.nan, -0.006, -0.016, -0.008, 0.017, 0.03, -0.001, 0.01, 0.017, 0.013, 0.002, 0.04, 0.013],
    'cs_shift': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, -0.005, -0.008, 0.006, -0.002, -0.002, 0.001, -0.009, -0.

In [53]:
df_test_all_effects_rho02 = pd.DataFrame({
    'true': [0.024, 0.001, 0.006, -0.0, 0.013, 0.016, 0.02, 0.007, 0.008, -0.002, -0.004, 0.004, -0.001, -0.012, -0.006, 0.005, 0.003],
    'optimal': [np.nan, 0.384, 0.376, 0.387, 0.373, 0.377, 0.364, 0.375, 0.02, 0.014, 0.004, 0.026, 0.004, -0.001, 0.014, 0.037, 0.018],
    'linear': [np.nan, np.nan, 0.003, -0.003, 0.011, -0.001, -0.0, 0.0, 0.017, 0.014, 0.01, 0.029, -0.021, 0.008, 0.019, 0.031, 0.027],
    'conditional': [np.nan, np.nan, np.nan, 0.005, -0.012, 0.003, -0.003, -0.002, -0.004, -0.003, -0.003, 0.023, 0.017, -0.014, -0.002, 0.016, 0.02],
    'shift': [np.nan, np.nan, np.nan, np.nan, 0.001, 0.005, -0.0, 0.012, 0.021, 0.002, 0.005, 0.007, 0.008, 0.004, -0.005, 0.002, -0.016],
    'cs': [np.nan, np.nan, np.nan, np.nan, np.nan, -0.006, -0.016, -0.008, 0.017, 0.03, -0.001, 0.01, 0.017, 0.013, 0.002, 0.04, 0.013],
    'cs_shift': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, -0.005, -0.008, 0.006, -0.002, -0.002, 0.001, -0.009, -0.012, 0.005, 0.0, -0.001],
    'conditional_ts': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, -0.005, -0.01, -0.015, 0.0, 0.009, -0.007, -0.008, 0.017, -0.002, -0.005],
    'conditional_cs': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.004, 0.01, 0.003, -0.01, 0.004, 0.006, 0.002, 0.008, 0.008],
    'Global_MLP': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.093, 0.162, 0.085, 0.087, 0.107, 0.165, 0.104, 0.111],
    '1D_Trans_TT': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.031, 0.129, 0.108, 0.162, 0.041, 0.128, 0.138],
    '1D_Trans_CC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.042, 0.045, 0.031, 0.103, 0.045, 0.048],
    '2D_Trans_TC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.221, 0.118, 0.044, 0.244, 0.24],
    '2D_Trans_TCTC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.112, 0.031, 0.216, 0.213],
    '1D_Trans_TT_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.027, 0.133, 0.145],
    '1D_Trans_CC_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.053, 0.048],
    '2D_Trans_TC_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.274],
})
df_test_all_effects_rho05 = pd.DataFrame({
    'true': [0.037, 0.003, 0.02, 0.023, 0.015, 0.01, 0.016, 0.011, 0.006, -0.007, -0.004, 0.01, -0.001, -0.008, 0.018, -0.003, 0.002],
    'optimal': [np.nan, 0.374, 0.398, 0.383, 0.38, 0.361, 0.38, 0.377, 0.047, 0.026, 0.037, 0.056, 0.105, 0.059, 0.035, 0.081, 0.101],
    'linear': [np.nan, np.nan, 0.003, -0.01, 0.008, -0.001, -0.002, -0.005, 0.05, 0.012, 0.053, 0.037, 0.109, 0.038, 0.039, 0.076, 0.128],
    'conditional': [np.nan, np.nan, np.nan, 0.018, 0.024, -0.005, 0.014, 0.004, -0.006, 0.003, -0.011, 0.053, 0.043, 0.006, 0.002, 0.058, 0.071],
    'shift': [np.nan, np.nan, np.nan, np.nan, 0.005, -0.002, -0.01, 0.0, 0.024, -0.008, 0.016, 0.016, 0.015, 0.014, 0.038, 0.014, 0.004],
    'cs': [np.nan, np.nan, np.nan, np.nan, np.nan, -0.012, -0.01, -0.0, 0.035, 0.047, 0.02, 0.048, 0.109, 0.054, 0.018, 0.057, 0.051],
    'cs_shift': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, -0.009, -0.009, 0.034, 0.011, 0.019, 0.006, 0.006, 0.009, -0.004, 0.006, 0.008],
    'conditional_ts': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.018, -0.006, -0.001, -0.0, -0.015, -0.002, 0.019, -0.001, 0.003, -0.003],
    'conditional_cs': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, -0.007, 0.006, 0.001, 0.003, -0.0, 0.016, 0.001, 0.002, 0.011],
    'Global_MLP': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.101, 0.157, 0.082, 0.097, 0.103, 0.179, 0.08, 0.099],
    '1D_Trans_TT': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.028, 0.106, 0.13, 0.171, 0.02, 0.141, 0.135],
    '1D_Trans_CC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.038, 0.039, 0.036, 0.1, 0.028, 0.039],
    '2D_Trans_TC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.244, 0.102, 0.038, 0.211, 0.201],
    '2D_Trans_TCTC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.14, 0.05, 0.281, 0.253],
    '1D_Trans_TT_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.034, 0.135, 0.152],
    '1D_Trans_CC_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.047, 0.033],
    '2D_Trans_TC_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.26],
})
df_test_all_effects_rho1 = pd.DataFrame({
    'true': [0.094, 0.026, 0.033, 0.049, 0.037, 0.021, 0.03, 0.052, 0.002, 0.02, 0.006, 0.015, 0.028, 0.013, 0.019, 0.024, 0.019],
    'optimal': [np.nan, 0.377, 0.384, 0.384, 0.378, 0.367, 0.383, 0.362, 0.104, 0.094, 0.072, 0.147, 0.131, 0.064, 0.105, 0.219, 0.13],
    'linear': [np.nan, np.nan, 0.015, -0.01, 0.002, -0.001, -0.007, -0.009, 0.06, 0.083, 0.082, 0.112, 0.09, 0.059, 0.069, 0.205, 0.071],
    'conditional': [np.nan, np.nan, np.nan, 0.009, -0.001, -0.002, 0.003, -0.007, 0.008, 0.006, 0.023, 0.108, 0.111, 0.0, 0.032, 0.154, 0.087],
    'shift': [np.nan, np.nan, np.nan, np.nan, 0.003, -0.004, 0.006, -0.004, 0.061, -0.007, 0.062, 0.023, 0.014, 0.008, 0.089, 0.021, 0.036],
    'cs': [np.nan, np.nan, np.nan, np.nan, np.nan, 0.002, 0.009, -0.009, 0.087, 0.116, 0.02, 0.15, 0.129, 0.083, 0.047, 0.212, 0.131],
    'cs_shift': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, -0.005, -0.012, 0.068, 0.015, 0.026, 0.003, 0.004, 0.009, 0.031, 0.001, 0.003],
    'conditional_ts': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, -0.003, 0.007, 0.023, -0.005, -0.011, 0.009, -0.002, 0.012, -0.003, 0.01],
    'conditional_cs': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, -0.017, 0.011, -0.018, 0.005, -0.012, 0.011, -0.004, -0.011, 0.004],
    'Global_MLP': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.114, 0.169, 0.108, 0.105, 0.101, 0.187, 0.124, 0.115],
    '1D_Trans_TT': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.034, 0.146, 0.148, 0.176, 0.042, 0.153, 0.14],
    '1D_Trans_CC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.04, 0.038, 0.028, 0.133, 0.075, 0.035],
    '2D_Trans_TC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.254, 0.157, 0.061, 0.291, 0.255],
    '2D_Trans_TCTC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.127, 0.051, 0.265, 0.266],
    '1D_Trans_TT_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.057, 0.148, 0.143],
    '1D_Trans_CC_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.072, 0.064],
    '2D_Trans_TC_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.256],
})
df_test_all_effects_rho2 = pd.DataFrame({
    'true': [0.197, 0.075, 0.069, 0.073, 0.058, 0.084, 0.079, 0.08, 0.037, 0.018, 0.02, 0.039, 0.036, 0.024, 0.013, 0.043, 0.035],
    'optimal': [np.nan, 0.367, 0.369, 0.375, 0.365, 0.38, 0.391, 0.389, 0.145, 0.108, 0.123, 0.196, 0.19, 0.09, 0.12, 0.262, 0.215],
    'linear': [np.nan, np.nan, -0.006, -0.009, -0.007, -0.013, 0.007, -0.001, 0.093, 0.134, 0.113, 0.172, 0.174, 0.11, 0.104, 0.243, 0.199],
    'conditional': [np.nan, np.nan, np.nan, -0.012, -0.009, 0.002, 0.007, 0.002, -0.002, 0.011, 0.012, 0.174, 0.155, 0.01, 0.025, 0.186, 0.189],
    'shift': [np.nan, np.nan, np.nan, np.nan, -0.008, -0.0, 0.003, 0.007, 0.101, 0.005, 0.125, -0.005, 0.015, -0.011, 0.118, 0.027, 0.013],
    'cs': [np.nan, np.nan, np.nan, np.nan, np.nan, -0.006, -0.004, -0.002, 0.101, 0.124, 0.028, 0.163, 0.146, 0.124, 0.029, 0.173, 0.159],
    'cs_shift': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.01, 0.006, 0.114, 0.007, 0.019, -0.003, 0.003, -0.004, 0.029, 0.0, -0.006],
    'conditional_ts': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.004, -0.014, -0.011, 0.023, 0.006, 0.009, -0.002, 0.01, 0.043, 0.002],
    'conditional_cs': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, -0.011, 0.014, 0.004, 0.011, 0.003, 0.013, 0.001, 0.023, 0.015],
    'Global_MLP': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.115, 0.158, 0.101, 0.106, 0.104, 0.195, 0.134, 0.13],
    '1D_Trans_TT': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.046, 0.15, 0.149, 0.187, 0.052, 0.162, 0.187],
    '1D_Trans_CC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.062, 0.062, 0.035, 0.124, 0.066, 0.067],
    '2D_Trans_TC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.292, 0.142, 0.076, 0.306, 0.305],
    '2D_Trans_TCTC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.144, 0.068, 0.302, 0.311],
    '1D_Trans_TT_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.04, 0.149, 0.171],
    '1D_Trans_CC_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.084, 0.081],
    '2D_Trans_TC_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.337],
})
df_test_all_effects_rho5 = pd.DataFrame({
    'true': [0.494, 0.2, 0.197, 0.174, 0.181, 0.191, 0.187, 0.19, 0.188, 0.126, 0.234, 0.316, 0.299, 0.127, 0.252, 0.324, 0.306],
    'optimal': [np.nan, 0.383, 0.378, 0.371, 0.384, 0.388, 0.381, 0.385, 0.39, 0.258, 0.476, 0.634, 0.593, 0.25, 0.506, 0.649, 0.606],
    'linear': [np.nan, np.nan, 0.007, -0.003, 0.006, -0.002, 0.008, 0.01, 0.252, 0.294, 0.289, 0.382, 0.363, 0.302, 0.29, 0.395, 0.368],
    'conditional': [np.nan, np.nan, np.nan, 0.007, 0.003, 0.007, -0.003, -0.011, 0.009, 0.032, 0.169, 0.318, 0.313, 0.022, 0.176, 0.334, 0.316],
    'shift': [np.nan, np.nan, np.nan, np.nan, -0.006, 0.005, -0.013, -0.003, 0.254, 0.002, 0.287, 0.323, 0.287, 0.008, 0.295, 0.32, 0.287],
    'cs': [np.nan, np.nan, np.nan, np.nan, np.nan, 0.002, 0.002, 0.012, 0.26, 0.329, 0.164, 0.375, 0.362, 0.295, 0.209, 0.368, 0.353],
    'cs_shift': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.008, 0.012, 0.251, 0.006, 0.197, 0.007, 0.002, 0.007, 0.208, 0.001, -0.001],
    'conditional_ts': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.014, 0.001, 0.005, 0.154, 0.24, 0.219, 0.002, 0.17, 0.2, 0.234],
    'conditional_cs': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.014, 0.021, 0.008, 0.048, 0.036, 0.031, 0.003, 0.115, 0.059],
    'Global_MLP': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.219, 0.34, 0.308, 0.292, 0.208, 0.356, 0.314, 0.307],
    '1D_Trans_TT': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.164, 0.304, 0.292, 0.338, 0.182, 0.301, 0.299],
    '1D_Trans_CC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.418, 0.392, 0.171, 0.414, 0.426, 0.405],
    '2D_Trans_TC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.677, 0.285, 0.454, 0.712, 0.688],
    '2D_Trans_TCTC': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.289, 0.434, 0.671, 0.658],
    '1D_Trans_TT_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.178, 0.293, 0.28],
    '1D_Trans_CC_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.452, 0.438],
    '2D_Trans_TC_sparse': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.683],
})